# Retail Demand Forecast — Clean Demo

**Student:** Nosirov Nodir  
**Group:** Tulpar

This short notebook loads the repository-contained final Random Forest, validates one raw feature object, and predicts next week's product demand. It does not require the original M5 dataset, processed Parquet files, or a local MLflow database.

## 1. Prepare a clean runtime

In Google Colab this cell clones the public repository and installs its tested dependencies. In a local clone it reuses the current repository.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/intelNod/retail-demand-forecasting.git"
IN_COLAB = "google.colab" in sys.modules

def find_repository():
    candidates = [Path.cwd(), Path.cwd() / "retail-demand-forecasting", Path("/content/retail-demand-forecasting")]
    for candidate in candidates:
        if (candidate / "models" / "final_model" / "MLmodel").exists():
            return candidate.resolve()
    return None

PROJECT_ROOT = find_repository()
if PROJECT_ROOT is None and IN_COLAB:
    subprocess.run(["git", "clone", REPOSITORY_URL], check=True)
    PROJECT_ROOT = find_repository()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository or run it in Google Colab.")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Repository ready:", PROJECT_ROOT.name)
print("Standalone final model: found")

Repository ready: retail-demand-forecasting
Standalone final model: found


## 2. Load one complete example

The 20 values describe past demand, known prices, and known calendar/SNAP information for the forecast week.

In [2]:
import json

with open(PROJECT_ROOT / "examples" / "predict_request.json", encoding="utf-8") as file:
    example = json.load(file)

print(json.dumps(example, indent=2))

{
  "sales_lag_1": 12,
  "sales_lag_2": 10,
  "sales_lag_4": 8,
  "sales_lag_8": 9,
  "sales_roll_mean_4": 10.5,
  "sales_roll_std_4": 1.8,
  "sales_roll_mean_8": 9.7,
  "sales_roll_std_8": 2.1,
  "zero_weeks_last_4": 0,
  "sell_price_mean": 2.49,
  "price_lag_1": 2.49,
  "price_change_pct": 0,
  "year": 2016,
  "month": 5,
  "week_of_year": 20,
  "quarter": 2,
  "event_days": 0,
  "has_event": 0,
  "snap_days": 3,
  "has_snap": 1
}


## 3. Validate and predict

The same reusable function is used by this notebook, the Flask API, and the browser UI.

In [3]:
from src.inference import load_final_model, predict_weekly_demand
from src.validation import FEATURE_NAMES

model = load_final_model()
prediction = predict_weekly_demand(example, model=model)
result = {
    "predicted_weekly_units": round(prediction, 4),
    "unit": "units_per_week",
    "model": "random_forest_final",
}

assert tuple(model.feature_names_in_) == FEATURE_NAMES
assert prediction >= 0
print(json.dumps(result, indent=2))

{
  "predicted_weekly_units": 10.4885,
  "unit": "units_per_week",
  "model": "random_forest_final"
}


## 4. Show safe failure on invalid input

A missing value is rejected before the model is called.

In [4]:
from src.validation import InputValidationError

invalid_example = dict(example)
invalid_example.pop("sales_lag_1")
try:
    predict_weekly_demand(invalid_example, model=model)
except InputValidationError as error:
    print("Expected validation error:", error)
else:
    raise AssertionError("Invalid input was not rejected.")

Expected validation error: Missing required fields: sales_lag_1.


## Result

The clean demo loaded the committed final model, produced a non-negative weekly forecast from a documented input, and rejected an invalid input without using hidden notebook state.